# 결측치와 전처리

독창적 가상 데이터와 자체 연습 문항입니다. 제공 샘플의 흐름을 참고했으며 원문 복제가 아닙니다.

14문항 · 파트 연습 · 힌트 사용 가능

CSV 두 개와 노트북을 같은 폴더에 두세요. 커널 재시작 → 전체 실행 → 저장한 뒤 사이트에 제출하세요. 각 답안 다음 검증 셀은 실행 결과를 검사합니다. 검증 셀을 바꾸면 점수 신뢰도가 사라집니다. 공식 점수 예측이 아닌 자체 연습 점수입니다.

데이터: ticket_id 식별번호, created_at 접수일시, issue_type 문의유형, contact_channel 채널, membership_tier 등급, tenure_months 가입개월, monthly_spend 월금액(원), previous_tickets 과거 문의 수, wait_minutes 대기분, past_satisfaction_score 과거만족도(1~5), escalated 이관여부(0/1), resolution_hours 해결시간(시간). resolution_hours는 분류 입력에서 제외합니다.

In [175]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, RobustScaler, LabelEncoder
from sklearn.metrics import accuracy_score, recall_score, f1_score, roc_auc_score, confusion_matrix, mean_absolute_error
SET_ID='part-prep'
DATA_FILE="practice.csv"
MEMBERS_FILE="members.csv"
MODEL_FILE="best_part-prep.keras"

from pathlib import Path
import json as _json
from IPython.display import display as _display
_raw = pd.read_csv(DATA_FILE)
_grade_results = []
def _filled(d):
    x=d.copy()
    cols=['wait_minutes','past_satisfaction_score','monthly_spend']
    x[cols]=x[cols].fillna(x[cols].median())
    x['contact_channel']=x.contact_channel.fillna(x.contact_channel.mode().iloc[0])
    return x
def _plot_ok(fig,h,k1,k2,title,ylabel,labels=None):
    a=fig.axes[0]
    ls=a.get_lines()
    labels=labels or [k1,k2]
    return (len(ls)==2 and a.get_title()==title and a.get_xlabel()=='Epoch' and a.get_ylabel()==ylabel
      and np.allclose(ls[0].get_ydata(),h.history[k1]) and np.allclose(ls[1].get_ydata(),h.history[k2])
      and a.get_legend() is not None and [t.get_text() for t in a.get_legend().get_texts()]==labels)
def _grade_q(number,checks):
    items=[]
    for label,expr in checks:
        try:
            ok=bool(eval(expr,globals()))
            items.append({'label':label,'passed':ok,'detail':'검사 통과' if ok else '실행 결과가 지시와 다릅니다. 변수명·조건·파라미터를 확인하세요.'})
        except Exception as e:
            items.append({'label':label,'passed':False,'detail':type(e).__name__+': '+str(e)[:180]})
    report={'schema':'aice-lab-v1','setId':SET_ID,'question':number,'checks':items}
    _grade_results.append(report)
    _display({'application/json':report},raw=True)
data = pd.read_csv(DATA_FILE)

## 1. 중복 제거 (7점)

data의 완전 중복 행을 제거하여 dedup에 저장하세요.

In [ ]:
# 여기에 답안코드를 작성하고 실행하세요.
dedup = data.drop_duplicates()


np.int64(0)

In [178]:
_grade_q(1, [['중복 제거 결과', 'dedup.equals(_raw.drop_duplicates())']])

## 2. 특정 결측 행 제거 (7점)

dedup에서 wait_minutes가 결측인 행만 제거하여 wait_known에 저장하세요.

In [181]:
# 여기에 답안코드를 작성하고 실행하세요.
wait_known = dedup.dropna(subset=["wait_minutes"])
wait_known["wait_minutes"].isna().sum()

np.int64(0)

In [182]:
_grade_q(2, [['subset 삭제', 'wait_known.equals(_raw.drop_duplicates().dropna(subset=["wait_minutes"]))']])

## 3. 모든 결측 행 제거 (7점)

dedup에서 하나라도 결측치가 있는 행을 제거하여 complete에 저장하세요.

In [183]:
# 여기에 답안코드를 작성하고 실행하세요.
complete = dedup.dropna()

In [184]:
_grade_q(3, [['전체 결측행 삭제', 'complete.equals(_raw.drop_duplicates().dropna())']])

## 4. 중앙값 대체 (7점)

dedup을 복사한 filled를 만들고 wait_minutes 결측치를 이 컬럼의 중앙값으로 채우세요.

In [185]:
# 여기에 답안코드를 작성하고 실행하세요.
filled = dedup.copy()

filled["wait_minutes"] = filled["wait_minutes"].fillna(filled["wait_minutes"].median())


In [186]:
_grade_q(4, [['중앙값 대체', 'filled.wait_minutes.equals(_raw.drop_duplicates().wait_minutes.fillna(_raw.drop_duplicates().wait_minutes.median()))']])

## 5. 최빈값 대체 (7점)

filled의 contact_channel 결측치를 최빈값 첫 번째 값으로 채우세요.

In [187]:
# 여기에 답안코드를 작성하고 실행하세요.
filled["contact_channel"] = filled["contact_channel"].fillna(filled["contact_channel"].mode().iloc[0])

In [188]:
_grade_q(5, [['최빈값 대체', 'filled.contact_channel.equals(_raw.drop_duplicates().contact_channel.fillna(_raw.drop_duplicates().contact_channel.mode().iloc[0]))']])

## 6. 이상치 필터 (7점)

filled에서 wait_minutes가 300 이상인 행을 제거하여 filtered에 저장하세요.

In [190]:
# 여기에 답안코드를 작성하고 실행하세요.
filtered = filled[filled["wait_minutes"] < 300]

In [191]:
_grade_q(6, [['정확한 행 유지', 'set(filtered.ticket_id)==set(_raw.drop_duplicates().loc[lambda d:d.wait_minutes.fillna(d.wait_minutes.median())<300,"ticket_id"])']])

## 7. 인덱스 초기화 (7점)

filtered의 인덱스를 초기화하세요. 옛 인덱스를 컬럼으로 남기지 마세요.

In [192]:
# 여기에 답안코드를 작성하고 실행하세요.
filtered = filtered.reset_index(drop=True)

In [193]:
_grade_q(7, [['0부터 연속 인덱스', 'np.array_equal(filtered.index,np.arange(len(filtered)))'], ['옛 인덱스 없음', '"index" not in filtered.columns']])

## 8. 컬럼 제거 (7점)

filtered에서 ticket_id와 created_at을 제거한 reduced를 만드세요.

In [195]:
# 여기에 답안코드를 작성하고 실행하세요.
reduced = filtered.drop(columns=["ticket_id","created_at"])
reduced.head()

,issue_type,contact_channel,membership_tier,tenure_months,monthly_spend,previous_tickets,wait_minutes,past_satisfaction_score,escalated,resolution_hours
0,Product,Chat,Silver,48,52600.0,2,19.5,4.4,0,6.61
1,Payment,Chat,Basic,13,106900.0,2,9.8,3.8,0,5.69
2,Account,Phone,Silver,51,68400.0,4,9.5,2.8,0,9.97
3,Delivery,Chat,Gold,21,82400.0,1,30.5,4.3,0,10.05
4,Payment,Phone,Basic,17,30900.0,0,4.3,3.8,0,1.98


In [196]:
_grade_q(8, [['두 컬럼 제거', 'reduced.equals(filtered.drop(columns=["ticket_id","created_at"]))']])

## 9. 범주 통합 (7점)

filtered를 복사한 merged를 만들고 Payment와 Account를 Account/Billing으로 합치세요.

In [198]:
# 여기에 답안코드를 작성하고 실행하세요.
merged = filtered.copy()

merged.loc[
    merged["issue_type"].isin(["Payment", "Account"]),
    "issue_type"
] = "Account/Billing"

merged["issue_type"].value_counts()

issue_type
Account/Billing    253
Delivery           125
Product            114
Technical          103
Name: count, dtype: int64

In [199]:
_grade_q(9, [['범주 통합', 'merged.issue_type.equals(filtered.issue_type.replace({"Payment":"Account/Billing","Account":"Account/Billing"}))']])

## 10. 수치형 일괄 대체 (7점)

merged의 monthly_spend, past_satisfaction_score 결측치를 각각의 중앙값으로 채우세요.

In [202]:
# 여기에 답안코드를 작성하고 실행하세요.

median_cols = ["monthly_spend","past_satisfaction_score"]

merged[median_cols] = merged[median_cols].fillna(merged[median_cols].median()) 

In [203]:
_grade_q(10, [['남은 수치 결측 0', 'merged[["monthly_spend","past_satisfaction_score"]].isna().sum().sum()==0'], ['중앙값 검증', 'np.allclose(merged.monthly_spend,filtered.monthly_spend.fillna(filtered.monthly_spend.median()))']])

## 11. 빈칸 문자열 변환 (7점)

["12", "unknown", "8"]을 Series로 만든 뒤 숫자로 변환하여 numeric_answer에 저장하세요. 변환 실패는 NaN.

In [204]:
# 여기에 답안코드를 작성하고 실행하세요.
numeric_answer = pd.Series(["12","unknown","8"])
numeric_answer = pd.to_numeric(numeric_answer, errors="coerce")

In [205]:
_grade_q(11, [['숫자와 NaN', 'numeric_answer.iloc[0]==12 and pd.isna(numeric_answer.iloc[1]) and numeric_answer.iloc[2]==8']])

## 12. merge (7점)

members.csv를 members에 읽고 dedup과 ticket_id 기준 왼쪽 병합하여 joined에 저장하세요.

In [206]:
# 여기에 답안코드를 작성하고 실행하세요.
members = pd.read_csv("./members.csv")
joined = pd.merge(dedup, members, how="left", on="ticket_id")

In [207]:
_grade_q(12, [['왼쪽 병합 결과', 'joined.equals(pd.merge(_raw.drop_duplicates(),pd.read_csv(MEMBERS_FILE),on="ticket_id",how="left"))']])

## 13. concat (7점)

dedup의 앞 3행과 뒤 3행을 세로로 이어 pieces에 저장하세요. 인덱스는 0~5.

In [210]:
# 여기에 답안코드를 작성하고 실행하세요.
pieces = pd.concat([dedup.head(3), dedup.tail(3)], ignore_index=True, axis=0)

In [211]:
_grade_q(13, [['concat 결과', 'pieces.equals(pd.concat([_raw.drop_duplicates().head(3),_raw.drop_duplicates().tail(3)],ignore_index=True))']])

## 14. 최종 검증 (9점)

merged의 전체 결측 개수를 final_missing에, 중복 수를 final_duplicates에 저장하세요.

In [213]:
# 여기에 답안코드를 작성하고 실행하세요.
final_missing = merged.isna().sum().sum()
final_duplicates = merged.duplicated().sum()


In [214]:
_grade_q(14, [['결측 없음', 'final_missing==0 and merged.isna().sum().sum()==0'], ['중복 없음', 'final_duplicates==0 and merged.duplicated().sum()==0']])

## 제출
검증 결과를 포함한 상태로 저장하세요. VS Code에서 실행이 중단되면 오류를 고친 뒤 다시 전체 실행하세요. 설치가 필요하면 사이트의 환경 설정 안내를 확인하세요.